# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook provides a step-by-step guide for loading, exploring, and processing a FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema and accessible at:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This will fetch the Croissant schema, making all entities available for programmatic access.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (not as a dict, but as a dataset object)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier (@id): {dataset.metadata.id}")
print(f"Fields: {getattr(dataset.metadata, 'keywords', [])}")

## 2. Data Overview
Review the dataset's record sets, fields, and their unique `@id` values. Referencing by `@id` ensures reproducible and standards-compliant data processing.

In [ ]:
# The FAIR² dataset contains multiple record sets.
# List available record sets and their @id.

record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- {rs.id}: {rs.name}")

# Let's review one record set (using its @id) to see available fields/columns.
# For example, let's select the main survey record set.
main_rs = None
for rs in record_sets:
    if "survey" in rs.name.lower() or "mental health" in rs.name.lower():
        main_rs = rs
        break
if not main_rs:
    # If not found, just use the first record set
    main_rs = record_sets[0]
    print(f"Defaulting to record set: {main_rs.id} ({main_rs.name})")

print(f"Fields and columns in '{main_rs.name}' (@id: {main_rs.id}):")
for field in main_rs.fields:
    print(f"  Field: {field.id} - {getattr(field, 'name', '')}")
    for column in field.columns:
        print(f"    Column: {column.id} ({column.name})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always reference record set and field `@id`s for reproducibility and traceability.

In [ ]:
# Extract data from each record set using the @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"{rs_id}: columns = {df.columns.tolist()}")

# Let's display the first 5 rows of the main survey dataset
main_rs_id = main_rs.id
if main_rs_id in dataframes:
    display_df = dataframes[main_rs_id]
    print(f"Columns in dataframes['{main_rs_id}']:")
    print(display_df.columns.tolist())
    display_df.head()
else:
    print(f"No records loaded for record set '{main_rs_id}'.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. All references use `@id` fields according to Croissant standards.

In [ ]:
# --- Example EDA ---
# Find a numeric field for further analysis (e.g., GAD-7 score, PHQ-9 score, age)

# Let's try common names that are likely present
numeric_field_id = None
candidate_fields = ["gad_7_score", "phq_9_score", "psq_score", "age"]
display_df = dataframes.get(main_rs_id, pd.DataFrame())

for col in display_df.columns:
    for f in candidate_fields:
        if f.lower() in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id:
        break

if numeric_field_id:
    print(f"Using '{numeric_field_id}' as numeric field for EDA.")
    threshold = 10
    # Filter records
    filtered_df = display_df[display_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, e.g., 'gender' or 'level_of_education'
    group_fields = ["gender", "level_of_education", "village"]
    group_field_id = None
    for gf in group_fields:
        for col in display_df.columns:
            if gf.lower() in col.lower():
                group_field_id = col
                break
        if group_field_id:
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("No groupable categorical field found.")
else:
    print("No numeric field found in main survey record set.")

## 5. Visualization
Visualize data distributions or relationships. Below is an example: histogram for a numeric score and boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(display_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field if present
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=display_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to access a FAIR² mental health survey dataset using Croissant standards and `mlcroissant`. Using entity `@id`s provides reproducible data analysis workflows and facilitates compliant FAIR data access. The exploratory analysis shows how basic mental health scores vary and can be grouped by demographic attributes, informing further research and public health interventions.